In [ ]:
# Core libraries for data handling
import numpy as np
import pandas as pd
import os

# Utilities for splitting and evaluating the model
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Preprocessing and modeling tools
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# For saving the trained pipeline
import joblib

In [ ]:
# Define the raw Cleveland dataset path (relative to project root)
BASE_DIR    = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA_PATH   = os.path.join(BASE_DIR, "Data_set", "processed.cleveland.data")
OUTPUT_DIR  = os.path.join(BASE_DIR, "outputs", "random_forest")
MODEL_DIR   = os.path.join(BASE_DIR, "models")

# Read the whole raw file as text because the raw format is token-based and spans multiple lines
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Split all whitespace-separated tokens into a list
tokens = raw_text.split()

# Build records where each patient row ends with the token "name"
records = []
current_row = []

for token in tokens:
    # "name" marks the end of one patient record in this raw file
    if token == "name":
        records.append(current_row)
        current_row = []
    else:
        # Convert token to float and keep numeric values
        current_row.append(float(token))

# Basic check to ensure rows were extracted
print("Number of patient records parsed:", len(records))
print("Example row length:", len(records[0]) if len(records) > 0 else 0)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8e in position 58079: invalid start byte

In [ ]:
# Convert parsed records into a DataFrame (raw attributes before "name")
raw_df = pd.DataFrame(records)

# Attribute positions from heart-disease.names (1-based -> converted to 0-based)
# 3 age, 4 sex, 9 cp, 10 trestbps, 12 chol, 16 fbs, 19 restecg,
# 32 thalach, 38 exang, 40 oldpeak, 41 slope, 44 ca, 51 thal, 58 num(target)
selected_indices = [2, 3, 8, 9, 11, 15, 18, 31, 37, 39, 40, 43, 50, 57]

# Human-readable column names for the selected attributes
selected_columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

# Keep only the selected 14 columns
df = raw_df.iloc[:, selected_indices].copy()
df.columns = selected_columns

# Replace raw missing indicator -9 with NaN for proper imputation later
df.replace(-9, np.nan, inplace=True)

# Convert multiclass diagnosis into binary classification:
# 0 means no heart disease, 1 means presence of heart disease
df["target"] = (df["num"] > 0).astype(int)

# Drop the original "num" column because "target" is now the prediction label
df.drop(columns=["num"], inplace=True)

# Show data snapshot and missing counts
print(df.head())
print("\nMissing values per column:\n", df.isnull().sum())
print("\nClass distribution:\n", df["target"].value_counts())

In [ ]:
# Separate features (X) and label (y)
X = df.drop(columns=["target"])
y = df["target"]

# Split data while preserving class ratio using stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build a pipeline:
# 1) Median imputation for missing values
# 2) Standardization for SVM
# 3) SVM classifier with probability enabled for ROC-AUC
svm_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=42))
])

# Train the pipeline on training data
svm_pipeline.fit(X_train, y_train)

print("Training completed.")

In [ ]:
# Predict class labels on test set
y_pred = svm_pipeline.predict(X_test)

# Predict class probabilities for ROC-AUC (use probability of positive class)
y_prob = svm_pipeline.predict_proba(X_test)[:, 1]

# Compute key classification metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

# Print metric summary
print("Baseline SVM Results")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

# Print confusion matrix and full classification report
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# Define parameter grid for SVM inside the pipeline
param_grid = {
    "svm__C": [0.1, 1, 10, 50],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__kernel": ["rbf", "linear"]
}

# Use stratified 5-fold CV for robust model selection
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Optimize for ROC-AUC because this is a medical classification problem
grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Fit grid search on training set only
grid_search.fit(X_train, y_train)

# Get the best model from CV
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

In [ ]:
# Predict with tuned model on test data
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

# Compute tuned model metrics
best_acc = accuracy_score(y_test, best_pred)
best_prec = precision_score(y_test, best_pred)
best_rec = recall_score(y_test, best_pred)
best_f1 = f1_score(y_test, best_pred)
best_auc = roc_auc_score(y_test, best_prob)

# Print tuned performance
print("Tuned SVM Results")
print(f"Accuracy : {best_acc:.4f}")
print(f"Precision: {best_prec:.4f}")
print(f"Recall   : {best_rec:.4f}")
print(f"F1-score : {best_f1:.4f}")
print(f"ROC-AUC  : {best_auc:.4f}")

# Persist tuned model pipeline to disk for reuse
joblib.dump(best_model, "models/svm_heart_disease_model.joblib")
print("Saved model to models/svm_heart_disease_model.joblib")

In [ ]:
# Create an example patient row using the same feature order as training columns
sample_patient = pd.DataFrame([{
    "age": 58,
    "sex": 1,
    "cp": 4,
    "trestbps": 130,
    "chol": 250,
    "fbs": 0,
    "restecg": 1,
    "thalach": 140,
    "exang": 1,
    "oldpeak": 2.0,
    "slope": 2,
    "ca": 1,
    "thal": 7
}])

# Predict class and probability using tuned model
sample_pred = best_model.predict(sample_patient)[0]
sample_prob = best_model.predict_proba(sample_patient)[0, 1]

# Display patient-level prediction
print("Predicted class (0=no disease, 1=disease):", int(sample_pred))
print("Predicted disease probability:", round(float(sample_prob), 4))